# Generate Text Embeddings using CLIP

This notebook generates CLIP text embeddings for all downloaded products.

In [1]:
import torch
import clip
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# Load CLIP Model

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)

print(device)

cpu


# Load Dataset

In [3]:
products_df = pd.read_csv("../data/processed/cleaned_electronics_products.csv")

products_df.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965,"₹10,999","₹18,999"
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,"113,956","₹18,999","₹19,999"
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,"90,304","₹1,999","₹2,299"
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,"24,863","₹15,999","₹24,999"
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,"113,956","₹18,999","₹19,999"


# Load Image Mapping

In [4]:
mapping_df = pd.read_csv("../data/embeddings/image_mapping.csv")

mapping_df.head()

,image_name
0,0.jpg
1,1.jpg
2,10.jpg
3,100.jpg
4,1000.jpg


# Extract Product Indices

In [5]:
mapping_df["product_index"] = (
    mapping_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .astype(int)
)

mapping_df.head()

,image_name,product_index
0,0.jpg,0
1,1.jpg,1
2,10.jpg,10
3,100.jpg,100
4,1000.jpg,1000


In [6]:
downloaded_products = products_df.iloc[
    mapping_df["product_index"]
].reset_index(drop=True)

downloaded_products.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965,"₹10,999","₹18,999"
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,"113,956","₹18,999","₹19,999"
2,10,"realme narzo 50A Prime (Flash Black, 4GB RAM+6...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81IPrkMDqV...,https://www.amazon.in/realme-Storage-Definitio...,4.0,"8,307","₹9,749","₹13,499"
3,100,"Dell MS116 1000Dpi USB Wired Optical Mouse, Le...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/61onAgKP5g...,https://www.amazon.in/Dell-MS116-1000DPI-Wired...,4.5,"34,095",₹299,₹650
4,1000,HUMBLE Phone Holder for Bike/Bicycle for Phone...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71lmn51l-D...,https://www.amazon.in/Humble-Holder-Bicycle-Ph...,3.6,"3,113",₹209,₹699


In [6]:
print(len(downloaded_products))

NameError: name 'downloaded_products' is not defined

# Generate Text Embeddings

In [7]:
text_embeddings = []

for product_name in tqdm(downloaded_products["name"], desc="Generating Text Embeddings"):

    text = clip.tokenize(
        [str(product_name)],
        truncate=True
    ).to(device)

    with torch.no_grad():
        embedding = model.encode_text(text)

    text_embeddings.append(
        embedding.cpu().numpy().flatten()
    )

NameError: name 'downloaded_products' is not defined

In [11]:
text_embeddings = np.array(text_embeddings)

print(text_embeddings.shape)

(1257, 512)


# Save Text Embeddings

In [12]:
np.save(
    "../data/embeddings/text_embeddings.npy",
    text_embeddings
)

# Verify Saved File

In [8]:
products_df = pd.read_csv("../data/processed/cleaned_electronics_products.csv")

In [9]:
mapping_df = pd.read_csv("../data/embeddings/image_mapping.csv")

In [10]:
mapping_df["product_index"] = (
    mapping_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .astype(int)
)

In [11]:
downloaded_products = products_df.iloc[
    mapping_df["product_index"]
].reset_index(drop=True)

In [12]:
print(len(downloaded_products))

9599


In [13]:
text_embeddings = []

for product_name in tqdm(downloaded_products["name"], desc="Generating Text Embeddings"):

    text = clip.tokenize(
        [str(product_name)],
        truncate=True
    ).to(device)

    with torch.no_grad():
        embedding = model.encode_text(text)

    text_embeddings.append(
        embedding.cpu().numpy().flatten()
    )

Generating Text Embeddings: 100%|██████████| 9599/9599 [07:57<00:00, 20.11it/s]


In [14]:
text_embeddings = np.array(text_embeddings)

print(text_embeddings.shape)

(9599, 512)


In [15]:
np.save(
    "../data/embeddings/text_embeddings.npy",
    text_embeddings
)

print("✅ Text embeddings saved successfully!")

✅ Text embeddings saved successfully!


In [16]:
loaded_text = np.load("../data/embeddings/text_embeddings.npy")

print(loaded_text.shape)

(9599, 512)
